In [0]:
from pyspark.sql.functions import ( col, lit, upper, lower, round, when, sum, avg, count, max, min, rank, dense_rank, row_number, lag, lead, udf, to_date, month, year, countDistinct ) 
from pyspark.sql.window import Window 
from pyspark.sql.types import ( StringType, IntegerType, DoubleType ) 
# ── Dataset 1: Q1 Sales ─────────────────────────────── 
q1_sales = [ (1, "S001", "P001", "North", 150, 500.0, "2024-01-15"), (2, "S002", "P002", "South", 200, 300.0, "2024-01-22"), (3, "S003", "P001", "East", 120, 500.0, "2024-02-10"), (4, "S004", "P003", "North", 180, 750.0, "2024-02-18"), (5, "S005", "P002", "West", 250, 300.0, "2024-02-25"), (6, "S006", "P003", "South", 100, 750.0, "2024-03-05"), (7, "S007", "P001", "West", 300, 500.0, "2024-03-12"), (8, "S008", "P004", "North", 90, 1200.0, "2024-03-20"), ] 
q1_cols = ["id","sale_id","product_id", "region","quantity","unit_price","sale_date"] 
q1_df = spark.createDataFrame(q1_sales, q1_cols) 

# ── Dataset 2: Q2 Sales ─────────────────────────────── 
q2_sales = [ (9, "S009", "P001", "North", 200, 500.0, "2024-04-08"), (10, "S010", "P002", "South", 150, 300.0, "2024-04-15"), (11, "S011", "P003", "East", 220, 750.0, "2024-05-02"), (12, "S012", "P004", "West", 80, 1200.0, "2024-05-18"), (13, "S013", "P001", "South", 310, 500.0, "2024-05-25"), (14, "S014", "P002", "North", 175, 300.0, "2024-06-10"), (15, "S015", "P003", "West", 140, 750.0, "2024-06-20"), (16, "S016", "P004", "East", 95, 1200.0, "2024-06-28"), ]
q2_df = spark.createDataFrame(q2_sales, q1_cols) 
# ── Dataset 3: Products ─────────────────────────────── 
products = [ ("P001", "laptop", "Electronics", 500.0), ("P002", "headphones", "Electronics", 300.0), ("P003", "office chair","Furniture", 750.0), ("P004", "monitor", "Electronics", 1200.0), ] 
prod_cols = ["product_id","product_name", "category","standard_price"] 
prod_df = spark.createDataFrame(products, prod_cols)
 # ── Dataset 4: Region Targets ───────────────────────── 
targets = [ ("North", 150000.0), ("South", 120000.0), ("East", 100000.0), ("West", 130000.0), ] 
tgt_cols = ["region","sales_target"] 
tgt_df = spark.createDataFrame(targets, tgt_cols) 
print("✅ All datasets created!") 
display(q1_df) 
display(q2_df) 
display(prod_df) 
display(tgt_df)

✅ All datasets created!


id,sale_id,product_id,region,quantity,unit_price,sale_date
1,S001,P001,North,150,500.0,2024-01-15
2,S002,P002,South,200,300.0,2024-01-22
3,S003,P001,East,120,500.0,2024-02-10
4,S004,P003,North,180,750.0,2024-02-18
5,S005,P002,West,250,300.0,2024-02-25
6,S006,P003,South,100,750.0,2024-03-05
7,S007,P001,West,300,500.0,2024-03-12
8,S008,P004,North,90,1200.0,2024-03-20


id,sale_id,product_id,region,quantity,unit_price,sale_date
9,S009,P001,North,200,500.0,2024-04-08
10,S010,P002,South,150,300.0,2024-04-15
11,S011,P003,East,220,750.0,2024-05-02
12,S012,P004,West,80,1200.0,2024-05-18
13,S013,P001,South,310,500.0,2024-05-25
14,S014,P002,North,175,300.0,2024-06-10
15,S015,P003,West,140,750.0,2024-06-20
16,S016,P004,East,95,1200.0,2024-06-28


product_id,product_name,category,standard_price
P001,laptop,Electronics,500.0
P002,headphones,Electronics,300.0
P003,office chair,Furniture,750.0
P004,monitor,Electronics,1200.0


region,sales_target
North,150000.0
South,120000.0
East,100000.0
West,130000.0


 You are a Data Engineer at a retail company. The company has sales data from Q1 and Q2 2024. The business team needs a complete sales analytics report. Your job: → Ingest raw sales data from multiple sources → Clean and validate it → Enrich with product and region information → Build analytics — rankings, trends, comparisons → Save final report as Delta table → Answer business questions using Spark SQL

In [0]:
q1_df.printSchema()
q2_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- sale_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- sale_date: string (nullable = true)

root
 |-- id: long (nullable = true)
 |-- sale_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- sale_date: string (nullable = true)



In [0]:
q1_df.count()

8

In [0]:
q2_df.count()


8

In [0]:
prod_df.count()

4

In [0]:
tgt_df.count()

4

In [0]:
from pyspark.sql.functions import count,when,col


q1_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in q1_df.columns
]).show()


+---+-------+----------+------+--------+----------+---------+
| id|sale_id|product_id|region|quantity|unit_price|sale_date|
+---+-------+----------+------+--------+----------+---------+
|  0|      0|         0|     0|       0|         0|        0|
+---+-------+----------+------+--------+----------+---------+



In [0]:
from pyspark.sql.functions import to_date

q1_df = q1_df.withColumn("sale_date",to_date("sale_date") )

In [0]:
q2_df =q2_df.withColumn("sale_date",to_date("sale_date"))

In [0]:
q2_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- sale_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- sale_date: date (nullable = true)



In [0]:
q1_df.dropDuplicates()

DataFrame[id: bigint, sale_id: string, product_id: string, region: string, quantity: bigint, unit_price: double, sale_date: date]

In [0]:
q1_df = q1_df.dropna()

In [0]:
sales_df = q1_df.unionByName(q2_df)
sales_df.count()
sales_df.printSchema()


root
 |-- id: long (nullable = true)
 |-- sale_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- sale_date: date (nullable = true)



In [0]:
sales_df.count()

16

In [0]:
# Revenue = Quantity × Unit Price

sales_df = sales_df.withColumn("revenue", col("quantity")* col("unit_price"))

In [0]:
sales_df = sales_df.join (
    prod_df,
    on = "product_id",
    how="left")

In [0]:
sales_df.show()

+----------+---+-------+------+--------+----------+----------+--------+------------+-----------+--------------+
|product_id| id|sale_id|region|quantity|unit_price| sale_date| revenue|product_name|   category|standard_price|
+----------+---+-------+------+--------+----------+----------+--------+------------+-----------+--------------+
|      P001|  1|   S001| North|     150|     500.0|2024-01-15| 75000.0|      laptop|Electronics|         500.0|
|      P002|  2|   S002| South|     200|     300.0|2024-01-22| 60000.0|  headphones|Electronics|         300.0|
|      P001|  3|   S003|  East|     120|     500.0|2024-02-10| 60000.0|      laptop|Electronics|         500.0|
|      P003|  4|   S004| North|     180|     750.0|2024-02-18|135000.0|office chair|  Furniture|         750.0|
|      P002|  5|   S005|  West|     250|     300.0|2024-02-25| 75000.0|  headphones|Electronics|         300.0|
|      P003|  6|   S006| South|     100|     750.0|2024-03-05| 75000.0|office chair|  Furniture|        

In [0]:
%sql 
show tables;


database,tableName,isTemporary
default,all_regions_sales,false
default,employee_rankings,false
default,employees,false
default,employees_clean,false
default,employees_enriched,false
default,employees_partitioned,false
default,high_earners,false
default,sales_analytics,false


In [0]:
%sql 
SHOW TABLES IN default;

database,tableName,isTemporary
default,all_regions_sales,false
default,employee_rankings,false
default,employees,false
default,employees_clean,false
default,employees_enriched,false
default,employees_partitioned,false
default,high_earners,false
default,sales_analytics,false


In [0]:
%whos

Variable        Type                 Data/Info
----------------------------------------------
DataFrame       type                 <class 'pyspark.sql.dataframe.DataFrame'>
DoubleType      DataTypeSingleton    <class 'pyspark.sql.types.DoubleType'>
IntegerType     DataTypeSingleton    <class 'pyspark.sql.types.IntegerType'>
StringType      type                 <class 'pyspark.sql.types.StringType'>
Window          type                 <class 'pyspark.sql.window.Window'>
avg             function             <function avg at 0xff1dae05bb00>
col             function             <function col at 0xff1dae05a340>
count           function             <function count at 0xff1dae05b880>
countDistinct   function             <function countDistinct at 0xff1dae07b740>
dense_rank      function             <function dense_rank at 0xff1dae07a980>
lag             function             <function lag at 0xff1dae082840>
lead            function             <function lead at 0xff1dae082980>
lit            

In [0]:
from pyspark.sql import DataFrame
[df_name for df_name, obj in globals().items()
 if isinstance(obj, DataFrame)]

['_',
 '__',
 'q1_df',
 'q2_df',
 'prod_df',
 'tgt_df',
 '_13',
 'sales_df',
 '_sqldf',
 '_20',
 '_21',
 '_55',
 '_62',
 '_63']

In [0]:
sales_df = sales_df.join(
    tgt_df,
    on="region",
    how="left"
)


In [0]:
sales_df.show()

+------+----------+---+-------+--------+----------+----------+--------+------------+-----------+--------------+------------+
|region|product_id| id|sale_id|quantity|unit_price| sale_date| revenue|product_name|   category|standard_price|sales_target|
+------+----------+---+-------+--------+----------+----------+--------+------------+-----------+--------------+------------+
| North|      P001|  1|   S001|     150|     500.0|2024-01-15| 75000.0|      laptop|Electronics|         500.0|    150000.0|
| South|      P002|  2|   S002|     200|     300.0|2024-01-22| 60000.0|  headphones|Electronics|         300.0|    120000.0|
|  East|      P001|  3|   S003|     120|     500.0|2024-02-10| 60000.0|      laptop|Electronics|         500.0|    100000.0|
| North|      P003|  4|   S004|     180|     750.0|2024-02-18|135000.0|office chair|  Furniture|         750.0|    150000.0|
|  West|      P002|  5|   S005|     250|     300.0|2024-02-25| 75000.0|  headphones|Electronics|         300.0|    130000.0|


STEP 8 — Add Business Columns : month, year, status, capital prod name, agg : all funct, window funct,
add question (top selling, best region, avg revenue, etc )


In [0]:
sales_df = sales_df.withColumn("month", month("sale_date"))

In [0]:
sales_df = sales_df.withColumn("year", year("sale_date"))

In [0]:
sales_df.show(3)

+------+----------+---+-------+--------+----------+----------+-------+------------+-----------+--------------+------------+-----+----+
|region|product_id| id|sale_id|quantity|unit_price| sale_date|revenue|product_name|   category|standard_price|sales_target|month|year|
+------+----------+---+-------+--------+----------+----------+-------+------------+-----------+--------------+------------+-----+----+
| North|      P001|  1|   S001|     150|     500.0|2024-01-15|75000.0|      laptop|Electronics|         500.0|    150000.0|    1|2024|
| South|      P002|  2|   S002|     200|     300.0|2024-01-22|60000.0|  headphones|Electronics|         300.0|    120000.0|    1|2024|
|  East|      P001|  3|   S003|     120|     500.0|2024-02-10|60000.0|      laptop|Electronics|         500.0|    100000.0|    2|2024|
+------+----------+---+-------+--------+----------+----------+-------+------------+-----------+--------------+------------+-----+----+
only showing top 3 rows


In [0]:
sales_df = sales_df.withColumn(
    "status",
    when(col("revenue") > 75000, "Excellent")
    .when(col("revenue") > 50000, "Good")
    .otherwise("Average")
)

In [0]:
sales_df.show()

+------+----------+---+-------+--------+----------+----------+--------+------------+-----------+--------------+------------+-----+----+---------+
|region|product_id| id|sale_id|quantity|unit_price| sale_date| revenue|product_name|   category|standard_price|sales_target|month|year|   status|
+------+----------+---+-------+--------+----------+----------+--------+------------+-----------+--------------+------------+-----+----+---------+
| North|      P001|  1|   S001|     150|     500.0|2024-01-15| 75000.0|      laptop|Electronics|         500.0|    150000.0|    1|2024|     Good|
| South|      P002|  2|   S002|     200|     300.0|2024-01-22| 60000.0|  headphones|Electronics|         300.0|    120000.0|    1|2024|     Good|
|  East|      P001|  3|   S003|     120|     500.0|2024-02-10| 60000.0|      laptop|Electronics|         500.0|    100000.0|    2|2024|     Good|
| North|      P003|  4|   S004|     180|     750.0|2024-02-18|135000.0|office chair|  Furniture|         750.0|    150000.0|

In [0]:
sales_df.groupBy("status").count().show()

+---------+-----+
|   status|count|
+---------+-----+
|     Good|    6|
|Excellent|    9|
|  Average|    1|
+---------+-----+



In [0]:
# sales_df.filter(col("status") == "Good").count().alias "Good_count".show()
sales_df.filter(col("status") == "Good") \
    .select(count("*").alias("Good_count")) \
    .show()

+----------+
|Good_count|
+----------+
|         6|
+----------+



In [0]:
sales_df= sales_df.withColumn("product_name", upper("product_name"))

In [0]:
region_sales_df = sales_df.groupBy("region").agg(sum("revenue").alias("Total_sales")).orderBy(col("Total_sales").desc())
sales_df.show()

region_sales_df.show()

+------+----------+---+-------+--------+----------+----------+--------+------------+-----------+--------------+------------+-----+----+---------+
|region|product_id| id|sale_id|quantity|unit_price| sale_date| revenue|product_name|   category|standard_price|sales_target|month|year|   status|
+------+----------+---+-------+--------+----------+----------+--------+------------+-----------+--------------+------------+-----+----+---------+
| North|      P001|  1|   S001|     150|     500.0|2024-01-15| 75000.0|      LAPTOP|Electronics|         500.0|    150000.0|    1|2024|     Good|
| South|      P002|  2|   S002|     200|     300.0|2024-01-22| 60000.0|  HEADPHONES|Electronics|         300.0|    120000.0|    1|2024|     Good|
|  East|      P001|  3|   S003|     120|     500.0|2024-02-10| 60000.0|      LAPTOP|Electronics|         500.0|    100000.0|    2|2024|     Good|
| North|      P003|  4|   S004|     180|     750.0|2024-02-18|135000.0|OFFICE CHAIR|  Furniture|         750.0|    150000.0|

In [0]:
Product_sales_df = sales_df.groupBy("product_name").agg(sum("revenue").alias("Total_sales")).orderBy(col("Total_sales").desc())
Product_sales_df.show()
                                    

+------------+-----------+
|product_name|Total_sales|
+------------+-----------+
|      LAPTOP|   540000.0|
|OFFICE CHAIR|   480000.0|
|     MONITOR|   318000.0|
|  HEADPHONES|   232500.0|
+------------+-----------+



In [0]:
Avg_qty_df = sales_df.groupBy("product_name").agg(avg(col("quantity")).alias("Avg_quantity")).orderBy(col("Avg_quantity").desc())
Avg_qty_df.show()

Avg_qty_df = sales_df.groupBy("product_name") \
    .agg(round (avg(col("quantity")),1).alias("Avg_quantity")) \
    .orderBy(col("Avg_quantity").desc())

Avg_qty_df.show()

+------------+-----------------+
|product_name|     Avg_quantity|
+------------+-----------------+
|      LAPTOP|            216.0|
|  HEADPHONES|           193.75|
|OFFICE CHAIR|            160.0|
|     MONITOR|88.33333333333333|
+------------+-----------------+

+------------+------------+
|product_name|Avg_quantity|
+------------+------------+
|      LAPTOP|       216.0|
|  HEADPHONES|       193.8|
|OFFICE CHAIR|       160.0|
|     MONITOR|        88.3|
+------------+------------+



In [0]:
sales_df.select("month", "product_name", "region", "revenue").agg(max("revenue")).show()

+------------+
|max(revenue)|
+------------+
|    165000.0|
+------------+



In [0]:
sales_df.orderBy(col("revenue").desc()) \
    .select("month", "product_name", "region", "revenue") \
    .show(1)

+-----+------------+------+--------+
|month|product_name|region| revenue|
+-----+------------+------+--------+
|    5|OFFICE CHAIR|  East|165000.0|
+-----+------------+------+--------+
only showing top 1 row


In [0]:
sales_df.select("revenue") \
    .distinct() \
    .orderBy(col("revenue").desc()) \
    .show(2)

+--------+
| revenue|
+--------+
|165000.0|
|155000.0|
+--------+
only showing top 2 rows


In [0]:
sales_df.orderBy(col("revenue").desc()) \
    .limit(2) \
    .orderBy(col("revenue").asc()) \
    .show(1)

+------+----------+---+-------+--------+----------+----------+--------+------------+-----------+--------------+------------+-----+----+---------+
|region|product_id| id|sale_id|quantity|unit_price| sale_date| revenue|product_name|   category|standard_price|sales_target|month|year|   status|
+------+----------+---+-------+--------+----------+----------+--------+------------+-----------+--------------+------------+-----+----+---------+
| South|      P001| 13|   S013|     310|     500.0|2024-05-25|155000.0|      LAPTOP|Electronics|         500.0|    120000.0|    5|2024|Excellent|
+------+----------+---+-------+--------+----------+----------+--------+------------+-----------+--------------+------------+-----+----+---------+
only showing top 1 row


In [0]:
# 3rd rank

from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

window = Window.orderBy(col("revenue").desc())

sales_df.withColumn(
    "rank",
    dense_rank().over(window)
).filter(col("rank") == 3).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------+----------+---+-------+--------+----------+----------+--------+------------+-----------+--------------+------------+-----+----+---------+----+
|region|product_id| id|sale_id|quantity|unit_price| sale_date| revenue|product_name|   category|standard_price|sales_target|month|year|   status|rank|
+------+----------+---+-------+--------+----------+----------+--------+------------+-----------+--------------+------------+-----+----+---------+----+
|  West|      P001|  7|   S007|     300|     500.0|2024-03-12|150000.0|      LAPTOP|Electronics|         500.0|    130000.0|    3|2024|Excellent|   3|
+------+----------+---+-------+--------+----------+----------+--------+------------+-----------+--------------+------------+-----+----+---------+----+



In [0]:
sales_df.agg(min("revenue")).show()

+------------+
|min(revenue)|
+------------+
|     45000.0|
+------------+

